In [ ]:
import argparse
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np


In [4]:
df = pd.read_excel("Concrete_Data.xls")

X = df.iloc[:, :-1]   # all columns except the last one
y = df.iloc[:, -1]    # the last column (strength)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


def categorize_strength(value):
    if value < 25.01:
        return 0  # low
    elif value < 55.02:
        return 1  # medium
    else:
        return 2  # high

y_classes = np.array([categorize_strength(val) for val in y])

# 1. Baseline Model

In [ ]:
#. The baseline will be a
model which compute the largest class on the training data, and predict everything in the
test-data as belonging to that class (corresponding to the optimal prediction by a logistic
regression model with a bias term and no features

In [11]:
num_low = np.sum(y_classes == 0)
num_medium = np.sum(y_classes == 1)
num_high = np.sum(y_classes == 2)

print(f"Class distribution:\n Low: {num_low}\n Medium: {num_medium}\n High: {num_high}")

Class distribution:
 Low: 295
 Medium: 588
 High: 147


In [ ]:
# sicne the medium class has the most samples, the baseline model will predict this class everytime 
predictions_baseline = np.full_like(y_classes, 1)

# 2. ANN

In [29]:
from sklearn.neural_network import MLPClassifier
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_classes, test_size=0.2, random_state=42) #20 pecent als test set

model = MLPClassifier(hidden_layer_sizes=(64, 32, 16), 
                      activation='relu', solver='adam',
                      #learning_rate='adaptive',
                      #learning_rate_init=0.005,
                      max_iter=500)
model.fit(X_train, y_train)
predictions_ann = model.predict(X_test)

In [30]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_test, predictions_ann))
print(classification_report(y_test, predictions_ann))   

[[ 52   5   0]
 [  7 114   2]
 [  0  11  15]]
              precision    recall  f1-score   support

           0       0.88      0.91      0.90        57
           1       0.88      0.93      0.90       123
           2       0.88      0.58      0.70        26

    accuracy                           0.88       206
   macro avg       0.88      0.81      0.83       206
weighted avg       0.88      0.88      0.87       206

